In [8]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

load_dotenv()
client = OpenAI()

In [9]:
SYSTEM_PROMPT = """
You are Mama Niyi, a warm and knowledgeable Nigerian home cook with 30 years of experience.
You specialise in traditional Nigerian cuisine — jollof rice, egusi soup, suya, moi moi, and more.

Guidelines:
- Give practical, step-by-step cooking guidance.
- Suggest substitutes for hard-to-find ingredients.
- Mention regional variations where relevant (Yoruba, Igbo, Hausa styles).
- Keep answers friendly and concise.
- If asked about non-Nigerian food, reply that you specialise in Nigerian cuisine.
"""

In [10]:
def chat(user_message: str, history: list) -> str:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT}
    ]

    # Add previous conversation
    messages.extend(history)

    # Add the latest user message
    messages.append({
        "role": "user",
        "content": user_message
    })

    response = client.chat.completions.create(
        model="gpt-4o",
        messages=messages,
        max_tokens=500,
        temperature=0.7
    )

    return response.choices[0].message.content

In [11]:
with gr.Blocks(theme=gr.themes.Soft()) as demo:

    gr.Markdown(
        "# 🍽️ Nigerian Recipe Assistant\n"
        "Ask Mama Niyi anything about Nigerian cooking!"
    )

    chatbot = gr.Chatbot(height=450)

    with gr.Row():
        msg = gr.Textbox(
            placeholder="e.g. How do I make jollof rice?",
            show_label=False,
            scale=9
        )
        send_btn = gr.Button("Send", scale=1, variant="primary")

    gr.ClearButton([msg, chatbot], value="Clear Chat")

    gr.Examples(
        examples=[
            "How do I make jollof rice with a smoky base?",
            "What's the difference between egusi and ogbono soup?",
            "Can I make moi moi without banana leaves?",
        ],
        inputs=msg,
    )

    # ----------------------------
    # Response handler
    # ----------------------------
    def respond(user_message, history):
        reply = chat(user_message, history)

        history.append({
            "role": "user",
            "content": user_message
        })

        history.append({
            "role": "assistant",
            "content": reply
        })

        return "", history

    # Events
    msg.submit(respond, inputs=[msg, chatbot], outputs=[msg, chatbot])
    send_btn.click(respond, inputs=[msg, chatbot], outputs=[msg, chatbot])

# ----------------------------
# Launch app
# ----------------------------
demo.launch()

/var/folders/z_/995lx11n10z0x5lktdyck_ym0000gn/T/ipykernel_4364/4132001643.py:1: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
